# **RSA & SHA-256**

## **概要**

基本の非対称鍵暗号であるRSA暗号と、ハッシュ値を求めるSHA-256を実装しました。

RSAは本来共通鍵暗号の鍵の送受信に使われ文字列には使われないようですが、今回は文字列の暗号化、復号化に使いました。

### RSA

桁数が大きい合成数の素因数分解が現実的な時間内で困難だという事を安全性の根拠に利用している公開鍵暗号の一つです。

任意の文字列を、任意の二つの素数から求められた公開鍵で暗号化し、秘密鍵で復号化します。

### SHA-256

アメリカ国家安全保障局によって設計されました。

作成したハッシュ関数とライブラリのハッシュ関数の出力を比較し、正しい値が出力されているかを確認します。

## **操作方法**

各コードブロックに書かれている説明を読み実行してください。

## **制作者のコメント**

pythonには符号無し32bit整数が存在しない為、オーバーフローの処理を用意しないといけないので、書いていて大変でした。
ビット単位で行なう処理はpythonで行なうもどではないと感じました。

rsa暗号は必要となる公開鍵が大きくなってしまったため、大きい素数を探すのに少し苦労しました。

関数は出来る限り副作用が発生しないように作成し、型は明示する様にしました。
型は無いより有る方が書きやすく、読み易いと感じました。
今度はhaskellなどで実装してみたいと思います。

どのように動いているか知りたい時は、自分で実装するのが簡単で身に付く方法だと思います。
次は楕円曲線暗号を実装したいです。

## **作成時間**

作成時間：**約10時間**


In [ ]:
# @title これはRSA暗号のコードです。 { display-mode: "form" }
# @markdown `prime_number_0`と`prime_number_1`には大きな素数を入力してください。

# @markdown `plain_text`には好きな文字列を入れてください。あまりにも長いと大きな素数の入力を求められます。

# @markdown 暗号化前と後が同じ値だと、`Match`が`True`になります。
from math import lcm, gcd

def generateKey(p: int,q: int) -> tuple[tuple[int, ...], ...]:
    N = p * q
    L = lcm(p-1,q-1)

    E = 65537
    if gcd(E, L) != 1:
        # もし65537が使えない場合は、3などの小さな値から探す
        E = 3
        while gcd(E, L) != 1:
            E += 2

    D = pow(E, -1, L) % L

    return (E, N), (D, N)

def encrypt(text:str, pubLic_key: tuple[int, ...]) -> int:
    E, N = pubLic_key
    text_byte = text.encode("utf-8")
    plain_number = int.from_bytes(text_byte,byteorder="big")
    if plain_number >= N:
        raise ValueError(f"テキストが長すぎます！\nデータのサイズ: {plain_number.bit_length()}ビット\n鍵のサイズ(N): {N.bit_length()}ビット\nもっと大きな素数が必要です。")
    encrypted_number = pow(plain_number,E,N)

    return encrypted_number

def decrypt(encrypted_number: int, private_key: tuple[int, ...]) -> str:
    D, N = private_key
    decrypted_number = pow(encrypted_number, D, N)
    decrypted_bytes = decrypted_number.to_bytes(( decrypted_number.bit_length() + 7) // 8,byteorder="big")

    return decrypted_bytes.decode("utf-8")

if __name__ == "__main__":
    prime_number_0 = 2**4423-1 # @param {type:"raw"}
    prime_number_1 = 2**3217-1 # @param {type:"raw"}
    public_key, private_key = generateKey(prime_number_0,prime_number_1)

    plain_text = "Hello, World!" # @param {type:"string"}

    encrypted_text = encrypt(plain_text,public_key)
    decrypted_text = decrypt(encrypted_text,private_key)

    print(f'''
    Public key: {public_key}
    Private key: {private_key}
    Encrypted text: {encrypted_text}
    Decrypted text: {decrypted_text}
    Match: {plain_text == decrypted_text}
    ''')



    Public key: (65537, 739889514749936722467014421952668006767623205872720578161780386639781274872652956831833191249411394719437958291487400058405897682620761393663453469674973051653283131103135080472260757815346451778394330540834973590979604760958039523635885976966725969503139887993673349157225819545837040666234485568218234715079192296966992696225867626772609484502284981971148703056621993316283783946553780685518065407211309146071348125393294190543782515564220112520339963418959734554047350166068383442040640995689607372035615702116696416895101410270195007102991747999946603424972997985591466434421291209398969393361937142400407935302259351230724916058960204954121195968472176132321018955487445610730071894719520336697518244254039075120856043912633034344595700029274789981556896381591547008648921916538034548306846873213725661903294088404294087131162004379608645513017979468851242363292166306650630874518454144789975740480222694362672291293883074288456823416722603495908438357310040176420713789

In [ ]:
# @title これはSHA-256のコードです。 { display-mode: "form" }

# @markdown `input`に好きな文章を入れてください。

# @markdown `Output`はこのコードによって求めたハッシュ値、`Expect`はライブラリによって求めたハッシュ値です。

# @markdown 同じ値が出力されると、`Match`が`True`になります。
from functools import reduce

INITIAL_H = (
    0x6a09e667, 0xbb67ae85, 0x3c6ef372, 0xa54ff53a, 0x510e527f, 0x9b05688c, 0x1f83d9ab, 0x5be0cd19
)
K = (
    0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5, 0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5,
    0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3, 0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174,
    0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc, 0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da,
    0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7, 0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967,
    0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13, 0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85,
    0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3, 0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070,
    0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5, 0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3,
    0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208, 0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2
)

mask32 = lambda x: x & 0xffffffff
rotr = lambda x, n: mask32(x >> n) | mask32(x << 32 -n)
shr = lambda x, n: (x >> n)
sigma0 = lambda x: rotr(x,7) ^ rotr(x,18) ^ shr(x,3)
sigma1 = lambda x: rotr(x,17) ^ rotr(x,19) ^ shr(x,10)
SIGMA0 = lambda x: rotr(x,2) ^ rotr(x,13) ^ rotr(x,22)
SIGMA1 = lambda x: rotr(x,6) ^ rotr(x,11) ^ rotr(x,25)
ch = lambda x, y, z: (x & y) ^ (~x & z)
maj = lambda x, y, z: (x & y) ^ (x & z) ^ (y & z)

def padding(message: bytes) -> bytes:
    bit_len = len(message) * 8
    return message + b'\x80' + b'\x00' * ((55 - len(message)) % 64) + bit_len.to_bytes(8,"big")

def build_scedule(block: bytes) -> tuple[int, ...]:
    w_initial = [ int.from_bytes( block[i:i+4] , "big") for i in range(0,64,4)] + [0] * 48
    def update_w(w:list[int], t:int) -> list[int]:
        w[t] = mask32( sigma1(w[t-2])+w[t-7]+sigma0(w[t-15])+w[t-16] )
        return  w
    return tuple(reduce(update_w,range(16,64),w_initial))

def step_round(H: tuple[int, ...], kw: tuple[int, ...]) -> tuple[int, ...]:
    a,b,c,d,e,f,g,h = H
    kt, wt = kw

    t1 = mask32( h+SIGMA1(e)+ch(e,f,g)+kt+wt )
    t2 = mask32( SIGMA0(a)+maj(a,b,c) )

    return mask32( t1+t2 ), a,b,c,mask32( d+t1 ),e,f,g

def process_block(h_status:tuple[int, ...],block: bytes) -> tuple[int, ...]:
    w = build_scedule(block)
    final_working_variables = reduce(step_round,zip(K,w),h_status)
    return tuple(mask32(x[0]+x[1]) for x in zip(h_status,final_working_variables) )

def sha256(message:bytes) -> str:
    padded = padding(message)
    block = (padded[i:i+64] for i in range(0,len(padded),64))
    final_h = reduce(process_block,block,INITIAL_H)
    return "".join(f"{x:08x}" for x in final_h)

if __name__ == "__main__":
    import hashlib

    input = "Answer to the Ultimate Question of Life, the Universe, and Everything"  # @param {type:"string"}
    test_input = input.encode("utf-8")

    result = sha256(test_input)
    official_result = hashlib.sha256(test_input).hexdigest()

    print(f"Output: {result}")
    print(f"Expect: {official_result}")
    print(f"Match:  {result == official_result}")


Output: ccd827897e8253333aca305b8fb9467b6a52e901b841e2028cde21330a7863c6
Expect: ccd827897e8253333aca305b8fb9467b6a52e901b841e2028cde21330a7863c6
Match:  True
